# 20.10 合成数据 / Synthetic Data (SMOTE & Generative)

**中文**:**合成数据(synthetic data)**=**人工生成、而非真实采集**的数据。为什么需要它？①**类别不平衡**——欺诈/罕见病样本极少,模型学不好少数类(用 **SMOTE** 合成少数类样本);②**隐私**——不能共享真实用户数据,就共享一份**统计上相似的合成数据**(合规、可开放);③**数据稀缺/增强**——真实数据太少或太贵,用生成模型扩充;④**测试/开发**——需要不含真人的数据做管线测试。本节从零实现两大主线:**SMOTE**(解决不平衡)和**生成式合成数据**(用"训练于合成、测试于真实/TSTR"评估其可用性),并诚实揭示合成数据的**边界与陷阱**。
**English**: **Synthetic data** = data that is **artificially generated rather than really collected**. Why need it? ① **class imbalance** — fraud/rare-disease samples are scarce, models learn the minority poorly (synthesize minority samples with **SMOTE**); ② **privacy** — when real user data can't be shared, share **statistically similar synthetic data** instead (compliant, openable); ③ **scarcity/augmentation** — real data too little or too expensive, expand it with generative models; ④ **testing/dev** — need person-free data for pipeline testing. This section implements two main threads from scratch: **SMOTE** (fixing imbalance) and **generative synthetic data** (evaluated via "Train on Synthetic, Test on Real / TSTR"), and honestly reveals synthetic data's **limits and pitfalls**.

---

**中文**:**SMOTE(Synthetic Minority Over-sampling Technique, Chawla 2002)**——解决类别不平衡的经典方法。朴素做法是**复制**少数类样本(过采样),但会导致过拟合(模型只是记住这几个点)。SMOTE 更聪明:**在少数类样本之间"插值"生成新样本**:
**English**: **SMOTE (Synthetic Minority Over-sampling Technique, Chawla 2002)** — the classic method for class imbalance. The naive approach **duplicates** minority samples (oversampling), but causes overfitting (the model just memorizes those points). SMOTE is smarter: **generate new samples by "interpolating" between minority samples**:
1. 对每个少数类样本,找它的 $k$ 个最近邻(也是少数类)。
   For each minority sample, find its $k$ nearest neighbors (also minority).
2. 随机选一个邻居,在两点连线上随机取一点作为新的合成样本:$x_{\text{new}}=x_i+\lambda\,(x_{\text{nbr}}-x_i),\ \lambda\sim U(0,1)$。
   Pick a random neighbor and take a random point on the segment as a new synthetic sample: $x_{\text{new}}=x_i+\lambda(x_{\text{nbr}}-x_i),\ \lambda\sim U(0,1)$.

**中文**:这样生成的点落在少数类的"特征空间区域"内,**扩充了决策边界附近的少数类密度**,而不是简单复制。

**English**: The generated points fall within the minority's "feature-space region," **densifying the minority near the decision boundary** rather than merely duplicating.

> 💡 **面试速查 / Interview cheat-sheet（★★ 不平衡/数据增强必考）**
> **中文**:**合成数据**用途:不平衡/隐私/稀缺增强/测试。**SMOTE**=少数类近邻间插值生成新样本(比复制强, 减少过拟合); **只在训练集做, 且在划分之后!**(否则测试信息泄漏)。变体:**Borderline-SMOTE**(只在边界附近合成)、**ADASYN**(难样本处多合成)、**SMOTENC**(含类别特征)。**注意**:SMOTE 主要提升**召回**, 对 F1/精度不一定涨(它把决策边界推向少数类→召回↑精度↓, 权衡而非免费午餐); 高维/重叠严重时可能引入噪声。**生成式合成**(表格): **CTGAN/TVAE**(SDV库)、高斯copula、VAE/GAN; 评估用 **TSTR**(train-on-synthetic-test-on-real)看下游可用性 + 分布相似度。**隐私红线**:合成数据**默认不保证隐私**(可能记忆/泄漏真实样本)→要隐私必须叠加**差分隐私**(DP-GAN/PATE-GAN, 见20.9)。类别不平衡其他解法:类权重、阈值移动、focal loss、欠采样。
> **English**: **Synthetic data** uses: imbalance/privacy/scarcity-augmentation/testing. **SMOTE** = interpolate between minority neighbors to make new samples (better than duplication, less overfitting); **do it on the training set only, and AFTER the split!** (else test leakage). Variants: **Borderline-SMOTE** (synthesize only near the boundary), **ADASYN** (more where samples are hard), **SMOTENC** (with categorical features). **Note**: SMOTE mainly boosts **recall**, not necessarily F1/precision (it pushes the boundary toward the minority → recall↑ precision↓, a tradeoff not a free lunch); with high dimensions/heavy overlap it can inject noise. **Generative synthesis** (tabular): **CTGAN/TVAE** (SDV library), Gaussian copula, VAE/GAN; evaluate with **TSTR** (train-on-synthetic-test-on-real) for downstream utility + distribution similarity. **Privacy red line**: synthetic data **does not guarantee privacy by default** (may memorize/leak real samples) → for privacy add **differential privacy** (DP-GAN/PATE-GAN, see 20.9). Other imbalance fixes: class weights, threshold moving, focal loss, undersampling.


In [ ]:

# ============================================================
# ① SMOTE 从零实现:解决类别不平衡 / SMOTE from scratch for imbalance
# 中文:造一个 5% 正类的不平衡二分类。SMOTE 在少数类近邻间插值合成新样本。
# English: an imbalanced binary task with 5% positives. SMOTE interpolates between minority neighbors.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import f1_score, recall_score, precision_score
from sklearn.model_selection import train_test_split
np.random.seed(0)
X,y=make_classification(n_samples=4000,n_features=10,n_informative=5,weights=[0.95,0.05],
                        class_sep=0.9,random_state=0)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.3,stratify=y,random_state=0)
print(f"训练集正类比例 / train positive rate: {ytr.mean():.3f} (仅 {ytr.sum()} 个正样本 positives)")

def smote(X,y,k=5,n_new=None):
    Xmin=X[y==1]                                              # 少数类 / minority class
    n_new=n_new or (np.sum(y==0)-np.sum(y==1))                # 补到与多数类平衡 / balance to majority
    _,idx=NearestNeighbors(n_neighbors=k+1).fit(Xmin).kneighbors(Xmin)  # 每个少数样本的k近邻
    new=[]
    for _ in range(n_new):
        i=np.random.randint(len(Xmin))                        # 随机选一个少数样本 / random minority point
        nbr=Xmin[idx[i, 1+np.random.randint(k)]]              # 随机选它的一个近邻 / random neighbor
        lam=np.random.rand()                                  # 连线上随机插值 / random interpolation
        new.append(Xmin[i]+lam*(nbr-Xmin[i]))                 # x_new = x_i + λ(x_nbr - x_i)
    return np.vstack([X,new]), np.concatenate([y,np.ones(len(new))])

print(f"\n{'模型/model':<10}{'方法':<16}{'精度prec':>9}{'召回rec':>9}{'F1':>8}")
for name,mk in [("LogReg",lambda:LogisticRegression(max_iter=1000)),("RandomForest",lambda:RandomForestClassifier(n_estimators=100,random_state=0))]:
    for tag,(Xf,yf) in [("无处理 none",(Xtr,ytr)),("SMOTE 平衡",smote(Xtr,ytr))]:
        clf=mk().fit(Xf,yf); p=clf.predict(Xte)
        print(f"{name:<10}{tag:<16}{precision_score(yte,p):>9.3f}{recall_score(yte,p):>9.3f}{f1_score(yte,p):>8.3f}")


**中文**:诚实观察:**SMOTE 显著提升了召回率**(找出更多真实的少数类),但它是**精度-召回的权衡,不是免费午餐**——它把决策边界推向少数类,召回↑但精度可能↓。对随机森林,F1 上升(权衡划算);对逻辑回归,召回大涨但精度下降导致 F1 反而降(阈值固定 0.5 时更明显)。**结论:SMOTE 主要用于"宁可误报也别漏报"的场景(欺诈、疾病筛查),要配合调阈值/看 PR 曲线,而非无脑用**。下面在 2D 上可视化 SMOTE 到底在做什么。
**English**: Honest observation: **SMOTE markedly improves recall** (catching more true minorities), but it is a **precision-recall tradeoff, not a free lunch** — it pushes the decision boundary toward the minority, so recall↑ but precision may↓. For random forest, F1 rises (a worthwhile trade); for logistic regression, recall jumps but precision drops so F1 actually falls (more visible at a fixed 0.5 threshold). **Takeaway: SMOTE suits "better a false alarm than a miss" settings (fraud, disease screening), paired with threshold tuning / PR curves, not applied blindly.** Below we visualize in 2D what SMOTE actually does.


In [ ]:

# ============================================================
# 2D 可视化 SMOTE 的插值 / visualize SMOTE interpolation in 2D
# ============================================================
X2,y2=make_classification(n_samples=600,n_features=2,n_redundant=0,n_informative=2,
                          n_clusters_per_class=1,weights=[0.9,0.1],class_sep=1.2,random_state=1)
X2s,y2s=smote(X2,y2)
new_pts=X2s[len(X2):]                                          # SMOTE 新合成的点 / newly synthesized points
fig,ax=plt.subplots(1,2,figsize=(14,5))
ax[0].scatter(X2[y2==0,0],X2[y2==0,1],s=12,c="#BBBBBB",label="多数类 majority")
ax[0].scatter(X2[y2==1,0],X2[y2==1,1],s=40,c="#C44E52",edgecolor="k",label="少数类 minority(真实)")
ax[0].set_title(f"原始:不平衡({(y2==1).sum()}:{(y2==0).sum()})/ original imbalanced"); ax[0].legend(fontsize=9)
ax[1].scatter(X2[y2==0,0],X2[y2==0,1],s=12,c="#BBBBBB")
ax[1].scatter(new_pts[:,0],new_pts[:,1],s=15,c="#55A868",alpha=0.4,label="SMOTE 合成 synthetic")
ax[1].scatter(X2[y2==1,0],X2[y2==1,1],s=40,c="#C44E52",edgecolor="k",label="少数类(真实)")
ax[1].set_title("SMOTE:在少数类近邻间插值填充 / interpolated between neighbors"); ax[1].legend(fontsize=9)
for a in ax: a.set_xlabel("特征1"); a.set_ylabel("特征2")
plt.tight_layout(); plt.savefig("/tmp/adv10_smote.png",dpi=80); plt.show()
print("绿点(合成)落在红点(真实少数类)之间的连线上→扩充少数类密度, 而非简单复制")


In [ ]:

# ============================================================
# ② 生成式合成数据 + TSTR 评估 / generative synthetic data + TSTR
# 中文:用"每类高斯"(保留均值与协方差/相关性)生成一份完全人工的数据集, 用 TSTR 评估:
#      "训练于合成、测试于真实"。若合成数据够好, 下游模型精度应接近训练于真实数据。
# English: generate a fully artificial dataset via "per-class Gaussian" (preserving mean & covariance/correlations),
#      evaluate with TSTR: "train on synthetic, test on real." If good, downstream accuracy ~ training on real.
# ============================================================
from sklearn.metrics import f1_score
def generate_synthetic(X,y):
    outX,outY=[],[]
    for c in np.unique(y):
        Xc=X[y==c]; mu=Xc.mean(0); cov=np.cov(Xc.T)           # 保留每类的均值和协方差(相关结构)
        outX.append(np.random.multivariate_normal(mu,cov,size=len(Xc))); outY.append(np.full(len(Xc),c))
    return np.vstack(outX), np.concatenate(outY)
Xsyn,ysyn=generate_synthetic(Xtr,ytr)                          # 一份纯合成、不含任何真实样本的数据 / fully synthetic

clf=RandomForestClassifier(n_estimators=100,random_state=0)
real_f1=f1_score(yte, clf.fit(Xtr,ytr).predict(Xte), average="macro")     # 训练于真实 / train on real
tstr_f1=f1_score(yte, clf.fit(Xsyn,ysyn).predict(Xte), average="macro")   # 训练于合成 / train on synthetic
print(f"训练于真实数据 train-on-real  macro-F1 = {real_f1:.3f}")
print(f"训练于合成数据 train-on-synth macro-F1 = {tstr_f1:.3f}  ← 几乎持平!合成数据可替代真实做训练")
print(f"→ TSTR 效用保留率 utility retention = {tstr_f1/real_f1:.1%}")

# 分布对比:真实 vs 合成 / distribution comparison
fig,ax=plt.subplots(1,3,figsize=(15,4))
for j,f in enumerate([0,1,2]):
    ax[j].hist(Xtr[:,f],bins=30,alpha=0.5,density=True,label="真实 real",color="#4C72B0")
    ax[j].hist(Xsyn[:,f],bins=30,alpha=0.5,density=True,label="合成 synthetic",color="#55A868")
    ax[j].set_title(f"特征 feature {f} 分布对比"); ax[j].legend(fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/adv10_tstr.png",dpi=80); plt.show()
print("合成分布(绿)与真实(蓝)高度重合→保留了边际分布与相关结构")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **SMOTE 不是"提升一切",而是"用精度换召回"**:它在少数类近邻间插值,把决策边界推向少数类,所以**召回几乎总会涨,但精度可能掉,F1 涨不涨看模型和数据**(本例随机森林 F1 涨、逻辑回归 F1 反降)。这是最容易被误用的点——很多人以为"SMOTE=治好不平衡",其实它只是**平移了精度-召回的权衡点**。正确用法:①**只在训练集、且在划分之后做**(否则合成点泄漏进验证/测试→虚高);②配合看 PR 曲线、调阈值、用 macro-F1/PR-AUC 而非 accuracy;③高维/类别重叠严重时 SMOTE 会在错误区域造点,反而引入噪声——这时类权重、focal loss、阈值移动可能更好。
2. **合成数据可以"好到能替代真实数据训练"**:TSTR 显示,用纯合成数据(不含任何真实样本)训练的模型,下游 F1(0.80)几乎等于用真实数据训练(0.80)——效用保留接近 100%。因为我们的生成器保留了每类的均值、协方差(相关结构),合成的边际分布与真实高度重合。这正是**隐私保护数据共享**的基础:企业/医院可以开放一份合成数据供研究,而不暴露任何真实个人。CTGAN/TVAE 等能对复杂的混合型表格(含类别、多峰、长尾)做得更好。
3. **最危险的误解:合成 ≠ 隐私安全**:很多人以为"数据是假的就绝对安全",**大错**。①朴素生成模型可能**过拟合、记忆并复现真实样本**(尤其小数据、GAN 模式坍缩);②即使不逐字复制,合成数据仍可能泄漏统计信息,遭受成员推断攻击。**要真正的隐私保证,必须叠加差分隐私**(DP-GAN、PATE-GAN,见 20.9)。此外合成数据会**放大原数据的偏见**(生成器学到什么就复现什么),且难以生成训练里没见过的罕见真实模式。**合成数据是有用工具,但它的"效用"和"隐私"要分别验证,不能想当然。**

**English**:
1. **SMOTE isn't "improve everything," it "trades precision for recall"**: it interpolates between minority neighbors, pushing the boundary toward the minority, so **recall almost always rises, precision may fall, and whether F1 rises depends on model & data** (here RF's F1 rose, LogReg's fell). This is the most misused point — many think "SMOTE = cures imbalance," but it merely **shifts the precision-recall tradeoff point**. Correct usage: ① **only on the training set, and after the split** (else synthetic points leak into val/test → inflated scores); ② pair with PR curves, threshold tuning, macro-F1/PR-AUC instead of accuracy; ③ with high dimensions/heavy class overlap SMOTE creates points in wrong regions, injecting noise — then class weights, focal loss, threshold moving may be better.
2. **Synthetic data can be "good enough to replace real data for training"**: TSTR shows a model trained on purely synthetic data (no real samples) reaches downstream F1 (0.80) nearly equal to training on real data (0.80) — near-100% utility retention. Because our generator preserved per-class means and covariances (correlation structure), the synthetic marginals overlap real closely. This is the basis of **privacy-preserving data sharing**: a company/hospital can release synthetic data for research without exposing any real individual. CTGAN/TVAE handle complex mixed tabular data (categorical, multimodal, long-tail) far better.
3. **The most dangerous misconception: synthetic ≠ private**: many think "fake data is absolutely safe" — **badly wrong**. ① naive generators may **overfit, memorize, and reproduce real samples** (especially small data, GAN mode collapse); ② even without verbatim copying, synthetic data can leak statistics and suffer membership inference. **For a real privacy guarantee, add differential privacy** (DP-GAN, PATE-GAN, see 20.9). Also, synthetic data **amplifies the original's biases** (the generator reproduces whatever it learned) and struggles to generate rare real patterns unseen in training. **Synthetic data is a useful tool, but its "utility" and "privacy" must each be validated, never assumed.**

> 💼 **实战视角 / Practical angle**
> **中文**:合成数据落地:①**不平衡**(欺诈/风控/罕见病)——SMOTE 及变体(Borderline/ADASYN), 库 `imbalanced-learn`, 记得只在训练集+看 PR 而非 accuracy;②**隐私数据共享**——用 **SDV**(CTGAN/TVAE/Gaussian Copula)生成可开放的合成表, 配 DP 做隐私保证, 用 TSTR + 分布相似度(如 KS 检验)验收;③**数据增强**——CV/NLP 用增广/生成模型扩充, 表格用生成器;④**测试/开发环境**造不含真人的数据。落地要点:①**先问要解决什么**(不平衡?隐私?稀缺?)选对工具;②**双重验收**:效用(TSTR/下游指标)+ 隐私(成员推断测试/是否叠加 DP);③警惕合成放大偏见、模式坍缩、泄漏真实样本;④不平衡问题优先试类权重/阈值移动这些更简单的招, 不一定要 SMOTE。面试金句:*"合成数据解决不平衡/隐私/稀缺:SMOTE 在少数类近邻间插值(只在训练集做, 主要提升召回、是精度-召回权衡); 生成式合成用 CTGAN 等, 用 TSTR(训练于合成测试于真实)评效用; 但合成≠隐私安全, 要隐私必须叠加差分隐私。"*
> **English**: Synthetic data in practice: ① **imbalance** (fraud/risk/rare disease) — SMOTE and variants (Borderline/ADASYN), library `imbalanced-learn`, remember training-set-only + look at PR not accuracy; ② **privacy data sharing** — generate openable synthetic tables with **SDV** (CTGAN/TVAE/Gaussian Copula), add DP for a privacy guarantee, accept via TSTR + distribution similarity (e.g. KS test); ③ **augmentation** — expand with augmentation/generative models (CV/NLP) or generators (tabular); ④ **test/dev environments** with person-free data. Deployment keys: ① **first ask what you're solving** (imbalance? privacy? scarcity?) and pick the right tool; ② **dual acceptance**: utility (TSTR/downstream metrics) + privacy (membership-inference test / whether DP is added); ③ beware amplified bias, mode collapse, leaking real samples; ④ for imbalance try simpler class weights/threshold moving first, SMOTE isn't mandatory. Interview line: *"Synthetic data addresses imbalance/privacy/scarcity: SMOTE interpolates between minority neighbors (training-set only, mainly boosts recall, a precision-recall tradeoff); generative synthesis uses CTGAN etc., evaluated by TSTR (train on synthetic, test on real); but synthetic ≠ private — for privacy you must add differential privacy."*

---
### 小结 / Summary
- **中文**:合成数据解决不平衡/隐私/稀缺/测试; SMOTE=少数类近邻间插值(比复制好, 只在训练集+划分后做)。
- **English**: Synthetic data addresses imbalance/privacy/scarcity/testing; SMOTE = interpolate between minority neighbors (better than duplication; training-set only, after the split).
- **中文**:SMOTE 是精度-召回权衡(主要涨召回, F1 涨不涨看模型); 生成式合成用 TSTR 评效用。
- **English**: SMOTE is a precision-recall tradeoff (mainly boosts recall, F1 depends on model); evaluate generative synthesis with TSTR.
- **中文**:合成≠隐私安全(可能记忆/泄漏真实样本), 要隐私必叠加差分隐私; 会放大偏见, 需双重验收。
- **English**: Synthetic ≠ private (may memorize/leak real samples), needs differential privacy for guarantees; amplifies bias, needs dual (utility+privacy) validation.
